# SC4020 Lab 1 - Similarity Search with ViT Embedding

## 0. Import and Configs

In [1]:
import os
import json
from PIL import Image
from pathlib import Path
import csv

import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

In [2]:
# Set Device to GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
DATA_FOLDER = "data"
DATASET_FOLDER = os.path.join(DATA_FOLDER,"circo")
ANNOTATION_FILE = os.path.join(DATASET_FOLDER,"annotations.json")
EMBEDDING_FILE = os.path.join(DATASET_FOLDER,"embeddings.csv")

## Helper Functions

In [4]:
def to_coco_filename(img_id):
    return f"{img_id:012d}.jpg"

def to_coco_id(filename):
    return int(filename.split(".")[0])

In [5]:
def lp_norm(embedding_1,embedding_2,p=2):
    embedding_1 = torch.tensor(embedding_1)
    embedding_2 = torch.tensor(embedding_2)

    distance = torch.norm(embedding_1 - embedding_2, p=p)

    return distance

In [6]:
def cosine_similarity(embedding_1, embedding_2):
    embedding_1 = torch.tensor(embedding_1)
    embedding_2 = torch.tensor(embedding_2)

    similarity = torch.nn.functional.cosine_similarity(
        embedding_1,
        embedding_2,
        dim=0
    )

    return similarity

##  ViT Model

In [7]:
weights = ViT_B_16_Weights.DEFAULT

model = vit_b_16(weights=weights)
model.heads = torch.nn.Identity()
model = model.to(DEVICE)
model.eval()

preprocess = weights.transforms()

In [8]:
def get_vit_embedding(model,img_file_path):
    image = Image.open(img_file_path).convert("RGB")

    x = preprocess(image)
    x = x.unsqueeze(0).to(DEVICE)  # add batch dimension
    
    with torch.no_grad():
        embedding = model(x)

    return embedding

## Obtain ViT Embedding

In [ ]:
id_embedding_pair = {}

for file in Path(DATASET_FOLDER).iterdir():
    if file.suffix.lower() in [".jpg", ".jpeg"]:
        embedding = get_vit_embedding(model,file).squeeze(0).tolist()
        id_embedding_pair[to_coco_id(file.name)] = embedding

In [ ]:
with open(EMBEDDING_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    
    writer.writerow(["image_id", "embedding"])
    
    for image_id, embedding in id_embedding_pair.items():
        writer.writerow([image_id, embedding])

## Reading Annotation

In [ ]:
with open(ANNOTATION_FILE, "r") as f:
    annotations = json.load(f)

ref_gt_pair = {}

for annotation in annotations:
    ref_id = annotation["reference_img_id"]
    gt_img_ids = annotation["gt_img_ids"]

    ref_gt_pair[ref_id] = gt_img_ids

## Example

In [ ]:
# TO DO: Calculate pairwise distance for reference to gt, and pairwise distance between reference and 5 random images not from gt